# Athletia Phase 3: every frame vs adaptive sampling, FP32 vs FP16

Subset: the first 8 videos of 12 Parquet files spread across the 50 (`train-00000`, `-00004`, ... `-00044`),
first 60 seconds of each (about 96 videos, 5,760 video-seconds). Three runs on both T4s, one after another:

1. `run_full.py` FP32 with `--dump-frames`: the reference (the brief's rule on every frame, identical to `predict()`)
2. `run_full.py --half`: FP16 speed, and its agreement with the FP32 reference
3. `run_skip.py --half`: adaptive sampling (K=5, 80/20, margin 0.30-0.50) speed and agreement

Then, on CPU: `tools/sweep.py` replays the skip rule from the FP32 per-frame detections for 90 settings.
Expected GPU time: about 50 minutes.

**Settings:** Accelerator GPU T4 x2, Internet on, Add-ons > Secrets > `HF_TOKEN` attached.
Run with **Save Version, Save & Run All**, then download `compare_outputs.zip` from the version's Output tab.

In [ ]:
OUT_DIR = "/kaggle/working/compare"
ARCHIVE_NAME = "compare_outputs"

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/mabdullahi7780/Alethia-Coding-Task-Muhammad-Abdullah-Irfan.git"
BRANCH = "main"
REPO_DIR = "/tmp/alethia"  # outside /kaggle/working, so the output holds only results
DATA_DIR = "/tmp/finevideo"  # Parquet files live here, never in /kaggle/working


def run(cmd, cwd=None, check=True, log=None, env=None):
    """Run a command, print it, optionally tee its output to a log file in OUT_DIR."""
    print("$", " ".join(cmd), flush=True)
    t0 = time.perf_counter()
    if log is None:
        res = subprocess.run(cmd, cwd=cwd, env=env)
    else:
        res = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
        text = res.stdout + res.stderr
        with open(f"{OUT_DIR}/{log}", "w") as fh:
            fh.write(text)
        print("\n".join(text.splitlines()[-25:]))
    print(f"[exit {res.returncode}, {time.perf_counter() - t0:.1f}s]", flush=True)
    if check and res.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return res


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nsubject %s"], cwd=REPO_DIR)

# Same versions as development; Kaggle's preinstalled torch is used (version recorded in env.json).
run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.216", "av==18.1.0",
     "huggingface_hub", "imageio-ffmpeg", "pytest"])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
import importlib.metadata as metadata
import platform

import torch


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


env = {
    "commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": metadata.version("ultralytics"),
    "av": metadata.version("av"),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "gpu_mem_gb": [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1) for i in range(torch.cuda.device_count())],
    "cpu_model": sh("lscpu | grep 'Model name' | sed 's/.*: *//'"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "disk_free_gb": {p: round(shutil.disk_usage(p).free / 2**30, 1) for p in ["/kaggle/working", "/tmp"]},
    "disk_type": sh("lsblk -d -o NAME,ROTA,SIZE,MODEL"),
    "os": sh("cat /etc/os-release | grep PRETTY_NAME"),
}
print(json.dumps(env, indent=2))
assert torch.cuda.device_count() >= 1, "No GPU: set the accelerator to GPU T4 x2"
with open(f"{OUT_DIR}/env.json", "w") as handle:
    json.dump(env, handle, indent=2)

In [ ]:
from kaggle_secrets import UserSecretsClient

# Read from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN). Never printed, never written to disk.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded:", os.environ["HF_TOKEN"].startswith("hf_"))

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

FILES = ['data/train-00000-of-01357.parquet', 'data/train-00004-of-01357.parquet', 'data/train-00008-of-01357.parquet', 'data/train-00012-of-01357.parquet', 'data/train-00016-of-01357.parquet', 'data/train-00020-of-01357.parquet', 'data/train-00024-of-01357.parquet', 'data/train-00028-of-01357.parquet', 'data/train-00032-of-01357.parquet', 'data/train-00036-of-01357.parquet', 'data/train-00040-of-01357.parquet', 'data/train-00044-of-01357.parquet']
TOTAL_VIDEO_SECONDS = 457200
t0 = time.perf_counter()
for f in FILES:
    hf_hub_download("HuggingFaceFV/finevideo", f, repo_type="dataset", local_dir=DATA_DIR)
INPUT = os.path.join(DATA_DIR, "data")
print(len(os.listdir(INPUT)), "files,", round(sum(os.path.getsize(os.path.join(INPUT, x)) for x in os.listdir(INPUT)) / 2**30, 2),
      "GiB in", round(time.perf_counter() - t0), "s")
YOLO("yolo11l.pt")  # official weights into the repo folder
WEIGHTS = os.path.join(REPO_DIR, "yolo11l.pt")

## Runs

In [ ]:
SUBSET = ["--input", INPUT, "--weights", WEIGHTS, "--videos-per-file", "8", "--max-seconds", "60", "--stats-every", "60"]
RUNS = {
    "full32": ["run_full.py", "--dump-frames", f"{OUT_DIR}/full32.frames"],
    "full16": ["run_full.py", "--half"],
    "skip16": ["run_skip.py", "--half"],
}
timing = {}
for name, args in RUNS.items():
    t0 = time.perf_counter()
    run([sys.executable, args[0], *args[1:], "--output", f"{OUT_DIR}/{name}.jsonl", *SUBSET], log=f"{name}_run.log")
    timing[name] = round(time.perf_counter() - t0, 1)
    run([sys.executable, "tools/check_results.py", f"{OUT_DIR}/{name}.jsonl"], log=f"{name}_check.log", check=False)
print(timing)

## Accuracy against the FP32 every-frame reference

In [ ]:
pairs = {"full16_vs_full32": ("full32", "full16"), "skip16_vs_full32": ("full32", "skip16"),
         "skip16_vs_full16": ("full16", "skip16")}
for name, (ref, pred) in pairs.items():
    run([sys.executable, "tools/eval.py", "--ref", f"{OUT_DIR}/{ref}.jsonl", "--pred", f"{OUT_DIR}/{pred}.jsonl",
         "--json", f"{OUT_DIR}/{name}.json"], log=f"{name}.log")

## Offline sweep of skip settings (CPU only)

In [ ]:
run([sys.executable, "tools/sweep.py", "--dump", f"{OUT_DIR}/full32.frames", "--ref", f"{OUT_DIR}/full32.jsonl",
     "--out", f"{OUT_DIR}/sweep.csv"], log="sweep.log")

## Summary

In [ ]:
summary = {"timing_s": timing}
for name in RUNS:
    st = json.load(open(f"{OUT_DIR}/{name}.jsonl.stats.json"))
    gpus = list(st["gpu"].values())
    frames = sum(g["frames"] for g in gpus)
    secs = sum(g["seconds"] for g in gpus)
    rate = secs / st["wall_s"]
    summary[name] = {
        "wall_s": st["wall_s"], "video_seconds": secs, "frames_inferred": frames,
        "frames_per_video_second": round(frames / max(secs, 1), 2),
        "video_seconds_per_s": round(rate, 2),
        "projected_50_files_h_2xT4": round(TOTAL_VIDEO_SECONDS / rate / 3600, 1),
        "gpu_idle_s": [round(g["idle_s"], 1) for g in gpus], "gpu_infer_s": [round(g["infer_s"], 1) for g in gpus],
        "escalated": sum(g["escalated_seconds"] for g in gpus), "sampled": sum(g["sampled_seconds"] for g in gpus),
    }
for name in pairs:
    r = json.load(open(f"{OUT_DIR}/{name}.json"))
    summary[name] = {k: r[k] for k in ["seconds", "object_f1", "people_exact", "people_within1", "missing_seconds",
                                       "worst_videos_people_exact"]}
print(json.dumps(summary, indent=2))
json.dump(summary, open(f"{OUT_DIR}/summary.json", "w"), indent=2)

In [ ]:
archive = shutil.make_archive(f"/kaggle/working/{ARCHIVE_NAME}", "zip", OUT_DIR)
print("Archive:", archive, os.path.getsize(archive), "bytes")
print(sorted(os.listdir(OUT_DIR)))